# Week 9 Lab. Fit GARCH and Price the Risk
**Time Series Analysis & Random Processes** · Graduate School of Data Science, Chonnam National University · notebook v6 (2026-10-27)

---

### 이 실습에서 하는 일 / What this lab does

참값을 아는 모의 수익률로 오늘 덱의 세 질문을 돌려 봅니다. $\alpha_1 = 0.1$, $\beta_1 = 0.85$ 로 수익률 2,000일을 만들고, GARCH(1,1) 이 그 값을 찾아내는지, 찾은 $\sigma_t$ 로 내일의 손실 문턱(VaR)과 열흘 뒤의 변동성을 어떻게 말하는지 봅니다. 순서는 덱 「In the Lab, Fit GARCH and Price the Risk」 의 표 그대로이고, 단계마다 **먼저 예측하고** 실행합니다.
*We run today's three questions on simulated returns whose truth we know. We generate 2,000 days with $\alpha_1 = 0.1$ and $\beta_1 = 0.85$, ask whether GARCH(1,1) finds those values, and turn the fitted $\sigma_t$ into tomorrow's loss threshold and a ten-day volatility forecast. The order is the table on "In the Lab, Fit GARCH and Price the Risk", and every step is **predict first**, then run.*

1. **오를지 내릴지 모른다면, 그래도 무엇을 예측할 수 있나? / If we cannot predict whether prices will rise or fall, what can we predict?**
2. **왜 크게 흔들리는 날은 연달아 오나? / Why do large price swings often come in streaks?**
3. **모형이 모든 극단적인 움직임을 설명할까? / Can our model explain every unusually large move?**

덱 「Three Questions About Tomorrow’s Market」 의 세 질문입니다. 4단계의 VaR 는 2번의 변동성 예측을 쓰는 응용이고, 3번은 선택 A 가 S&P 500 으로 답합니다.
*These are the three questions on "Three Questions About Tomorrow’s Market". The VaR of step 4 applies the volatility forecast of question 2, and Optional A answers question 3 with the S&P 500.*

| 단계 / Step | 먼저 예측 / Before running, predict | 실행하고 확인 / Then run and check | 짝 슬라이드 / Matching slides |
|---|---|---|---|
| 0 | 모의 자료. 한 줄 공식(TODO 1)으로 수익률을 만든다<br>Simulated data: generate returns with the one-line recursion (TODO 1) | 수익률과 참 $\sigma_t$ 를 그린다<br>Plot the returns and the true $\sigma_t$ | 「Let Yesterday’s Shock Set Today’s Variance」 · 「GARCH Adds Yesterday’s Variance」 · 「One Day of GARCH(1,1) by Hand」 |
| 1 | r 의 ACF 는 평평할까? r² 의 ACF 는?<br>Will the ACF of r be flat? And the ACF of r²? | 두 ACF 를 그리고 시차 1 값을 비교한다<br>Plot both ACFs. Compare the lag-1 values | 「Two ACFs of the Same Returns」 |
| 2 | 적합은 α₁ + β₁ 로 얼마를 줄까?<br>What will the fit give for α₁ + β₁? | arch_model 로 GARCH(1,1) 을 맞추고 참값 0.95 와 견준다<br>Fit GARCH(1,1) with arch_model. Compare with the true 0.95 | 「Fitting GARCH(1,1) to Ten Years of the S&P 500」 · 「How Long Does a Storm in Volatility Last?」 |
| 3 | 적합한 $\sigma_t$ 는 참 $\sigma_t$ 를 따라갈까?<br>Will the fitted $\sigma_t$ follow the true one? | 참 · 적합 변동성을 함께 그리고, 표준화 잔차(TODO 2)를 점검한다<br>Plot the true and fitted volatility together, then check the standardized residuals (TODO 2) | 「Did GARCH Catch the Clustering?」 |
| 4 | 내일의 99% VaR 은 얼마나 클까?<br>How big is tomorrow’s 99% VaR? | 마지막 $\sigma_t$ 로 계산한다(TODO 3)<br>Compute it from the last $\sigma_t$ (TODO 3) | 「How Much Could We Lose Tomorrow?」 |
| 5 | 열흘 뒤 변동성은 어디로 갈까?<br>Where will volatility go in 10 days? | 10일 앞을 예측한다(TODO 4). 어느 수준으로 다가가나?<br>Forecast 10 days ahead (TODO 4). Which level does it approach? | 「Where Will Volatility Go Next?」 · 「How Can Daily Variance Have One Long-Run Level?」 |
| 6 | 선택 하나를 바꾼다<br>Change one choice | β₁ 이나 표본 크기를 바꾸고 차이 하나를 설명한다<br>Change β₁ or the sample size and explain one difference | 「When Does a Volatility Shock Never Fade?」 |

선택 A 는 덱의 실제 자료(S&P 500 10년)로 「Fitting GARCH(1,1) to Ten Years of the S&P 500」 과 「How Often Do Extreme Days Happen?」 의 숫자를 그대로 다시 얻습니다.
*Optional A reproduces the deck's real-data numbers on "Fitting GARCH(1,1) to Ten Years of the S&P 500" and "How Often Do Extreme Days Happen?" from the same ten years of the S&P 500.*

### 쓰는 법 / How to use it

**`TODO` 가 네 개 있습니다.** 이 네 줄을 직접 채우는 것이 이 노트북의 핵심입니다. 넷 다 오늘 덱의 식 한 줄입니다.
*Four cells are marked **`TODO`**. Those four lines are the part worth doing by hand. Each is one formula from today's deck.*

> 각 `TODO` 는 `… = None` 한 줄입니다. `None` 자리에 코드를 넣으면 되고, 지울 줄은 없습니다. 채우기 전에 실행하면 `NotImplementedError` 와 안내문이 뜨며 멈춥니다. **고장이 아니라 정상입니다.** `TODO` 바로 아래의 **자가검증** 셀이 내 답을 검사하고, 그 아래에 접힌 **Solution (정답)** 셀이 있습니다. 먼저 해 본 뒤에만 여세요. 정답 셀을 돌렸다면 자가검증 셀을 다시 실행하세요.
> *Each `TODO` is one line that reads `… = None`. Replace the `None` with your code; nothing needs deleting. If you run the cell before that, it stops with a `NotImplementedError` and a message. **That is expected, not a broken notebook.** The **self-check** cell right below each `TODO` tests your answer, and below it is a collapsed **Solution (정답)** cell. Open it only after you have tried. If you ran the Solution cell, run the self-check again.*

> ### 먼저 **파일 > Drive에 사본 저장** / First, **File > Save a copy in Drive**
> 올려 둔 링크는 **읽기 전용**으로 열립니다. 사본을 먼저 저장하지 않으면 입력한 내용이 탭을 닫을 때 사라집니다.
> *The posted link opens **read-only**. Save your own copy first, or everything you type disappears when you close the tab.*
>
> **제출하지 않습니다.** 9주차 실습 시간(약 35분)용입니다. 시간 안에 다 못 하면 5 · 6단계는 집에서 이어서 하세요.
> *Nothing to submit. This is the Week 9 lab block (about 35 minutes). If time runs out, finish steps 5 and 6 at home.*

**GPU 가 없어도 됩니다.** 모든 셀이 몇 초 안에 끝납니다. 숫자는 시드를 고정해 두어 같은 컴퓨터에서 다시 돌리면 그대로 나오고, 다른 컴퓨터 · 다른 판의 arch 에서는 소수 셋째 자리가 다를 수 있습니다.
*No GPU needed; every cell finishes in seconds. The seed is fixed, so a re-run on the same computer gives the same numbers; another computer or another arch version can differ in the third decimal.*

---
### 준비 / Setup

아래 셀을 실행하면 라이브러리를 불러옵니다. 오늘은 `arch` 패키지(GARCH 적합)가 더 필요합니다. Colab 에 없으면 셀이 알아서 설치합니다(몇 초).
*Run the cell below to import the libraries. Today we also need the `arch` package (GARCH fitting); if Colab does not have it, the cell installs it (a few seconds).*

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import norm

try:
    import arch                                     # GARCH 적합 패키지 / the GARCH-fitting package
except ImportError:                                 # Colab 에 없으면 설치(판 고정) / install it if Colab lacks it (pinned version)
    %pip install -q arch==8.0.0
    import arch
from arch import arch_model

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (11, 3.4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
GREEN, GOLD, RED, GRAY = "#1E4A32", "#D9AE3A", "#B34F3F", "#9A9A9A"   # 덱과 같은 색 / the deck's colours

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
# Message shown when a TODO is still blank. It means the notebook is fine, not broken.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막힐 때 자가검증 셀 아래의 'Solution / 정답' 셀을 실행하고, 자가검증 셀을 다시 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace None above with your code and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell below the self-check, then run the self-check again and continue.\n"
)


def acf1(x):
    """시차 1 의 표본 자기상관 / the sample autocorrelation at lag 1."""
    return acf(np.asarray(x, dtype=float), nlags=1, fft=False)[1]


def lb_p(x, lags=10):
    """Ljung–Box 검정의 p 값(시차 1 … lags 를 한꺼번에) / the Ljung–Box p-value over lags 1 to `lags` together."""
    return float(acorr_ljungbox(np.asarray(x, dtype=float), lags=[lags])["lb_pvalue"].iloc[0])


def fmt_p(p):
    """p 값을 덱과 같은 꼴로 / a p-value written the way the deck does."""
    return "< 0.001" if p < 0.001 else f"= {p:.2f}"


Z01 = norm.ppf(0.01)                                # 표준정규의 1% 점 −2.33 / the 1% point of the standard normal, −2.33
print("arch", arch.__version__, "· statsmodels", statsmodels.__version__, "· z_0.01 =", round(Z01, 4))
# 기대값 / Expected   arch 8.0.0 · statsmodels 0.15.0 · z_0.01 = -2.3263   (판이 달라도 됩니다 / other versions are fine)

**준비 셀이 만든 도구 / Tools made by the setup cell.** 외울 것은 없습니다. 무엇을 계산하는지만 보세요.
*Nothing to memorize. Just know what each one computes.*

| 이름 / Name | 하는 일 / What it does |
|---|---|
| `acf1(x)` | x 의 시차 1 표본 자기상관. 2주차의 ACF 에서 시차 1 값 하나<br>The sample autocorrelation of x at lag 1, one value from the Week 2 ACF |
| `lb_p(x, lags=10)` | Ljung–Box 검정의 p 값. 시차 1 ~ 10 을 한꺼번에 봄(7주차 잔차 점검과 같은 검정)<br>The Ljung–Box p-value over lags 1 to 10 together (the same test as the Week 7 residual check) |
| `Z01` | 표준정규의 1% 점, −2.3263. 덱은 −2.33 으로 적습니다<br>The 1% point of the standard normal, −2.3263, written −2.33 on the deck |

---
### 0단계. 참값을 아는 모의 수익률 / Step 0. Simulated returns with a known truth

실제 시장에서는 $\alpha_1$, $\beta_1$ 의 참값을 아무도 모릅니다. 그래서 먼저 참값을 **우리가 정한** 수익률을 만들고, 적합이 그 값을 찾아내는지 봅니다. 찾아내면 실제 자료의 적합도 믿을 근거가 생깁니다.
*In a real market nobody knows the true $\alpha_1$ and $\beta_1$. So we first generate returns whose truth **we chose**, and ask whether the fit recovers it. If it does, we have a reason to trust the fit on real data.*

GARCH(1,1) 의 하루는 덱 「GARCH Adds Yesterday’s Variance」 의 한 줄입니다. 오늘의 분산은 기준선, 어제 소식의 몫, 어제 분산에서 넘어온 몫의 합입니다. 수익률은 그 분산의 제곱근에 새 충격을 곱한 것입니다.
*One day of GARCH(1,1) is the one line on "GARCH Adds Yesterday’s Variance": today's variance is a baseline, a share of yesterday's news and a share carried over from yesterday's variance. The return is the square root of that variance times a fresh shock.*

$$\sigma_t^2 = \alpha_0 + \alpha_1 r_{t-1}^2 + \beta_1 \sigma_{t-1}^2, \qquad r_t = \sigma_t \varepsilon_t, \quad \varepsilon_t \sim N(0, 1)$$

오늘의 참값: $\alpha_0 = 0.05$, $\alpha_1 = 0.1$, $\beta_1 = 0.85$ (단위는 퍼센트). 그러면 $\alpha_1 + \beta_1 = 0.95$, 장기 분산 $\alpha_0 / (1 - \alpha_1 - \beta_1) = 1$, 곧 장기 퍼짐 1%, 반감기 $\ln 0.5 / \ln 0.95 = 13.5$ 일입니다(「How Long Does a Storm in Volatility Last?」 의 둘째 줄).
*Today's truth: $\alpha_0 = 0.05$, $\alpha_1 = 0.1$, $\beta_1 = 0.85$ (in percent). So $\alpha_1 + \beta_1 = 0.95$, the long-run variance is $\alpha_0 / (1 - \alpha_1 - \beta_1) = 1$, a long-run spread of 1%, and the half-life is $\ln 0.5 / \ln 0.95 = 13.5$ days (row 2 of "How Long Does a Storm in Volatility Last?").*

**TODO 1 은 그 한 줄, 어제에서 오늘의 분산을 만드는 식입니다.** 모의 자료를 만드는 셀이 이 함수를 하루에 한 번씩 부릅니다.
*TODO 1 is that one line: today's variance from yesterday. The simulation cell calls this function once per day.*

In [ ]:
# ---------------------------------------------------------------- TODO 1 ----
# 다음 이름을 씁니다 / Names used below
#   omega       : 기준선 α₀ / the baseline α₀
#   alpha       : 어제 소식의 몫 α₁ / the share of yesterday's news, α₁
#   beta        : 어제 분산에서 넘어오는 몫 β₁ / the share carried over from yesterday's variance, β₁
#   r_prev      : 어제의 수익률 r_{t−1} / yesterday's return r_{t−1}
#   sigma2_prev : 어제의 분산 σ²_{t−1} / yesterday's variance σ²_{t−1}
#   sigma2      : 오늘의 분산 σ²_t   <- 여기가 빈칸
#                 today's variance σ²_t   <- this is the blank
#
# 참고 / Reference   x ** 2
#   x 의 제곱 / x squared.   예 / e.g.   3 ** 2  ->  9

def next_variance(omega, alpha, beta, r_prev, sigma2_prev):
    """어제의 수익률과 분산에서 오늘의 분산을 만든다 / Today's variance from yesterday's return and variance."""
    # TODO 1. 덱 「GARCH Adds Yesterday’s Variance」 의 식을 그대로 쓰세요. 기준선 + 어제 소식의 몫 + 어제 분산의 몫. 한 줄입니다.
    #         어제 소식은 어제 수익률의 제곱입니다. 부호는 들어가지 않습니다.
    # TODO 1. Write the formula on "GARCH Adds Yesterday’s Variance" as it is: baseline + a share of yesterday's news + a share of yesterday's variance. One line.
    #         Yesterday's news is yesterday's return squared; its sign does not enter.
    sigma2 = None
    if sigma2 is None:
        raise NotImplementedError(BLANK_MSG)

    return sigma2


# 한 번 불러 보기. 덱 「One Day of GARCH(1,1) by Hand」 의 큰 날: α₀ = 0.02, α₁ = 0.1, β₁ = 0.85, 어제 수익률 3, 어제 분산 1.
# Call it once, with the big day of "One Day of GARCH(1,1) by Hand": α₀ = 0.02, α₁ = 0.1, β₁ = 0.85, yesterday's return 3, yesterday's variance 1.
print("큰 날의 오늘 분산 / today's variance after the big day:", round(next_variance(0.02, 0.1, 0.85, 3.0, 1.0), 3))
# 기대값 / Expected
#   큰 날의 오늘 분산 / today's variance after the big day: 1.77     (0.02 + 0.1 × 9 + 0.85 × 1, 슬라이드와 같음 / as on the slide)
#   다른 값이 나와도 바로 아래 자가검증이 어디가 틀렸는지 알려 줍니다.
#   If you get another value, the self-check below tells you what went wrong.
# --------------------------------------------------------------------------

In [ ]:
# 자가검증 / Self-check   덱 「One Day of GARCH(1,1) by Hand」 의 두 날 / the two days on "One Day of GARCH(1,1) by Hand"
cases = [(3.0, 1.0, 1.77), (0.5, 1.0, 0.895), (-3.0, 1.0, 1.77)]
for r_prev, s2_prev, want in cases:
    got = next_variance(0.02, 0.1, 0.85, r_prev, s2_prev)
    assert abs(got - want) < 1e-9, (
        f"어제 수익률 {r_prev}, 어제 분산 {s2_prev} 에서 정답은 {want}, 결과는 {got:.4f} 입니다. "
        "0.02 + 0.1 × (어제 수익률)² + 0.85 × (어제 분산) 입니다. 어제 수익률은 제곱해서 넣고(+3 과 −3 은 같은 값), "
        "0.1 은 수익률의 제곱에, 0.85 는 분산에 곱합니다.\n"
        f"Yesterday's return {r_prev} and variance {s2_prev} should give {want}, but gave {got:.4f}. "
        "The line is 0.02 + 0.1 × (yesterday's return)² + 0.85 × (yesterday's variance): square the return (+3 and −3 give the same value), "
        "put 0.1 on the squared return and 0.85 on the variance.")
print("통과. 어제가 크면(어느 쪽이든) 오늘의 분산이 크고, 어제 분산의 0.85 는 오늘로 넘어옵니다.")
print("Passed. A big day yesterday, either way, makes today's variance big, and 0.85 of yesterday's variance carries over.")

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 실행한 뒤에는 바로 위의 자가검증 셀을 다시 실행하세요. 자가검증은 지금 메모리에 있는 것을 검사합니다.
# After running it, run the self-check cell just above again. The self-check tests whatever is in memory now.

def next_variance(omega, alpha, beta, r_prev, sigma2_prev):
    """어제의 수익률과 분산에서 오늘의 분산을 만든다 / Today's variance from yesterday's return and variance."""
    sigma2 = omega + alpha * r_prev ** 2 + beta * sigma2_prev   # 빈칸이었던 줄. 기준선 + 소식의 몫 + 넘어온 몫
                                                                 # the blank: baseline + a share of the news + the carried-over share
    if sigma2 is None:
        raise NotImplementedError(BLANK_MSG)

    return sigma2


print("큰 날의 오늘 분산 / today's variance after the big day:", round(next_variance(0.02, 0.1, 0.85, 3.0, 1.0), 3))
# 기대값 / Expected
#   큰 날의 오늘 분산 / today's variance after the big day: 1.77

In [ ]:
# 모의 수익률 2,000일. 처음 500일은 출발값의 영향을 지우려고 버립니다 / 2,000 simulated days; the first 500 are dropped to forget the starting value
OMEGA, ALPHA, BETA = 0.05, 0.10, 0.85                   # 참값 / the truth
N, BURN = 2000, 500
rng = np.random.default_rng(7)                          # 난수 고정 / fix the randomness
eps = rng.standard_normal(N + BURN)                     # 매일의 새 충격 ε_t / the fresh shock ε_t of every day
sigma2 = np.empty(N + BURN)
r = np.empty(N + BURN)
sigma2[0] = OMEGA / (1 - ALPHA - BETA)                  # 장기 분산에서 출발 / start at the long-run variance
r[0] = np.sqrt(sigma2[0]) * eps[0]
for t in range(1, N + BURN):
    sigma2[t] = next_variance(OMEGA, ALPHA, BETA, r[t - 1], sigma2[t - 1])   # TODO 1 의 한 줄 / the TODO 1 line
    r[t] = np.sqrt(sigma2[t]) * eps[t]                  # r_t = σ_t ε_t
r, sigma_true = r[BURN:], np.sqrt(sigma2[BURN:])        # 수익률과 참 변동성, 2,000일 / the returns and the true volatility, 2,000 days
days = np.arange(N)

print(f"장기 퍼짐(참값) / long-run spread (truth): {np.sqrt(OMEGA / (1 - ALPHA - BETA)):.2f} %")
print(f"표본의 퍼짐(표준편차) / sample spread (standard deviation): {r.std():.3f} %")
print(f"참 σ_t 의 범위 / range of the true σ_t: {sigma_true.min():.2f} % to {sigma_true.max():.2f} %")
# 기대값 / Expected
#   장기 퍼짐 / long-run spread: 1.00 %
#   표본의 퍼짐 / sample spread : 1.003 %     장기 값 근처 / near the long-run value
#   참 σ_t 의 범위 / range     : 0.65 % to 2.61 %     조용한 날과 폭풍이 섞여 있습니다 / calm days and storms, mixed

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax1.plot(days, r, color=GREEN, lw=0.6); ax1.set_ylabel("return (%)"); ax1.set_title("Simulated GARCH(1,1) returns: big days come in bunches", fontsize=10)
ax2.plot(days, sigma_true, color=GOLD, lw=1.0); ax2.axhline(1.0, color=GRAY, ls="--", lw=0.8); ax2.set_ylabel(r"true $\sigma_t$ (%)"); ax2.set_xlabel("day")
ax2.set_title("The true volatility behind them (dashed: the long-run level 1.0)", fontsize=10)
plt.tight_layout(); plt.show()

**What to notice.** 위 그림의 큰 날들이 몰려 있는 것은 우리가 넣은 $\beta_1 = 0.85$ 때문입니다. 어제 분산의 85% 가 오늘로 넘어오니 폭풍은 하루에 끝나지 않습니다. 아래 그림이 그 폭풍의 원인인 참 $\sigma_t$ 이고, 실제 시장에서는 이 선을 볼 수 없습니다. GARCH 적합이 하는 일은 수익률만 보고 이 선을 되찾는 것입니다.
*The big days in the top panel bunch up because of the $\beta_1 = 0.85$ we put in: 85% of yesterday's variance carries into today, so a storm does not end in one day. The bottom panel is the true $\sigma_t$ behind it, a line nobody sees in a real market. The job of a GARCH fit is to recover that line from the returns alone.*

---
### 1단계. 두 ACF / Step 1. Two ACFs

**코드를 돌리기 전에 예측하세요.** 수익률 r 의 ACF 는 평평할까요? 제곱 수익률 r² 의 ACF 는? 시차 1 에서 어느 쪽이 더 클까요? 덱 「Two ACFs of the Same Returns」 에서 S&P 500 은 −0.10 과 0.21 이었습니다.
*Predict before running. Will the ACF of the returns r be flat? And the ACF of the squared returns r²? Which is larger at lag 1? On "Two ACFs of the Same Returns" the S&P 500 gave −0.10 and 0.21.*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your prediction here. This cell does not need to be run.
my_prediction_1 = ""

In [ ]:
acf_r = acf(r, nlags=20, fft=False)[1:]                   # 시차 1 ~ 20 / lags 1 to 20
acf_r2 = acf(r ** 2, nlags=20, fft=False)[1:]
band = 2 / np.sqrt(N)                                     # 2주차의 ±2/√n 띠 / the ±2/√n band of Week 2
print(f"시차 1 의 ACF / ACF at lag 1:   r {acf_r[0]:+.3f}    r² {acf_r2[0]:+.3f}    (띠 / band ±{band:.3f})")
print(f"Ljung–Box(10) p:   r {fmt_p(lb_p(r))}    r² {fmt_p(lb_p(r ** 2))}")
# 기대값 / Expected
#   시차 1 의 ACF / ACF at lag 1:   r +0.041    r² +0.259    (띠 / band ±0.045)
#   Ljung–Box(10) p:   r = 0.27    r² < 0.001
#   r 은 띠 안팎을 오가는 작은 막대, r² 은 큰 막대가 천천히 줄어듭니다. 방향에는 기억이 거의 없고 크기에는 기억이 있습니다.
#   r gives small bars around the band; r² gives large bars that fade slowly. Little memory in the direction, memory in the size.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, vals, title in ((ax1, acf_r, "ACF of the returns r"), (ax2, acf_r2, "ACF of the squared returns r²")):
    ax.bar(np.arange(1, 21), vals, color=GREEN, width=0.6)
    ax.axhline(band, color=GOLD, ls="--", lw=0.8); ax.axhline(-band, color=GOLD, ls="--", lw=0.8)
    ax.set_title(title, fontsize=10); ax.set_xlabel("lag")
ax1.set_ylabel("correlation"); plt.tight_layout(); plt.show()

**What to notice.** 덱 「Two ACFs of the Same Returns」 와 같은 모양입니다. 수익률 자체는 2주차 띠 안팎의 작은 막대뿐이라 선형으로는 내일 값을 예측할 수 없고, 제곱 수익률은 큰 막대가 천천히 줄어듭니다. 이것이 오늘 모형의 입장권입니다. 「Common Mistakes With Volatility Models」 의 둘째 줄대로, 이 점검 없이 GARCH 를 맞추지 않습니다.
*The same picture as "Two ACFs of the Same Returns". The returns themselves show only small bars around the Week 2 band, so tomorrow's value cannot be predicted linearly; the squared returns show large bars that fade slowly. That is the ticket to today's model. As row 2 of "Common Mistakes With Volatility Models" says, do not fit GARCH without this check.*

---
### 2단계. GARCH(1,1) 적합 / Step 2. Fit GARCH(1,1)

**코드를 돌리기 전에 예측하세요.** `arch_model` 로 맞추면 $\hat\alpha_1 + \hat\beta_1$ 은 얼마가 나올까요? 참값 0.95 와 얼마나 가까울까요? 2,000일이면 충분할까요?
*Predict before running. What will the fit give for $\hat\alpha_1 + \hat\beta_1$? How close to the true 0.95? Are 2,000 days enough?*

덱 「Fitting GARCH(1,1) to Ten Years of the S&P 500」 과 같은 호출입니다. 평균은 상수 하나, 오차는 정규분포, 적합은 최대우도입니다. `arch` 는 $\alpha_0$ 를 `omega` 로 찍습니다(「Reading GARCH(p, q) in Software」).
*The same call as on "Fitting GARCH(1,1) to Ten Years of the S&P 500": a constant mean, normal errors, maximum likelihood. `arch` prints $\alpha_0$ as `omega` ("Reading GARCH(p, q) in Software").*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your prediction here. This cell does not need to be run.
my_prediction_2 = ""

In [ ]:
am = arch_model(r, mean="Constant", vol="GARCH", p=1, q=1, dist="normal")   # 상수 평균 + GARCH(1,1) + 정규 오차 / constant mean + GARCH(1,1) + normal errors
res = am.fit(disp="off")                                                     # 최대우도 적합. disp="off" 는 반복 출력을 끈다 / maximum likelihood; disp="off" silences the iterations
est, se = res.params, res.std_err
truth = {"mu": 0.0, "omega": OMEGA, "alpha[1]": ALPHA, "beta[1]": BETA}
print("            추정값 / estimate   표준오차 / std. error   참값 / truth")
for name in ("mu", "omega", "alpha[1]", "beta[1]"):
    print(f"{name:9s}   {est[name]:8.3f}        {se[name]:8.3f}          {truth[name]:5.2f}")
a_hat, b_hat = est["alpha[1]"], est["beta[1]"]
persist = a_hat + b_hat
print(f"\nα₁ + β₁ = {persist:.3f}   (참값 / truth 0.95)")
print(f"반감기 / half-life  ln 0.5 / ln(α₁ + β₁) = {np.log(0.5) / np.log(persist):.1f} days   (참값 / truth 13.5)")
print(f"장기 분산 / long-run variance  ω / (1 − α₁ − β₁) = {est['omega'] / (1 - persist):.3f}   (참값 / truth 1.00)")
# 기대값 / Expected
#               추정값 / estimate   표준오차 / s.e.   참값 / truth
#   mu          -0.012       0.021      0.00
#   omega        0.042       0.015      0.05
#   alpha[1]     0.079       0.017      0.10
#   beta[1]      0.877       0.028      0.85
#   α₁ + β₁ = 0.956   반감기 15.4 days   장기 분산 0.954
#   참값과의 차이가 표준오차 한두 개 안입니다. 2,000일로 α₁ 은 ±0.02, β₁ 은 ±0.03 정도까지만 안다는 뜻입니다.
#   Each estimate is within one or two standard errors of the truth: with 2,000 days we know α₁ to about ±0.02 and β₁ to about ±0.03.

**What to notice.** 적합은 참값을 찾았지만 정확히는 아닙니다. $\hat\alpha_1 + \hat\beta_1 = 0.956$ 은 0.95 와 표준오차 안에서 같고, 반감기는 15 일(참값 13.5 일)입니다. 덱 「When Does a Volatility Shock Never Fade?」 의 표에서 보듯 이 합이 1 에 가까울수록 작은 추정 오차가 반감기와 장기 분산에서는 큰 차이가 됩니다. 0.956 과 0.95 의 반감기 차이가 벌써 2 일입니다.
*The fit found the truth, but not exactly: $\hat\alpha_1 + \hat\beta_1 = 0.956$ agrees with 0.95 within its standard error, and the half-life is 15 days (truth 13.5). As the table on "When Does a Volatility Shock Never Fade?" shows, the closer this sum is to 1, the more a small estimation error matters for the half-life and the long-run variance: 0.956 against 0.95 already moves the half-life by 2 days.*

---
### 3단계. 참 변동성과 적합 변동성 / Step 3. True and fitted volatility

**코드를 돌리기 전에 예측하세요.** 적합한 $\hat\sigma_t$ 는 참 $\sigma_t$ 를 따라갈까요? 폭풍의 꼭대기에서도 따라갈까요?
*Predict before running. Will the fitted $\hat\sigma_t$ follow the true $\sigma_t$? Even at the peak of a storm?*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your prediction here. This cell does not need to be run.
my_prediction_3 = ""

In [ ]:
sigma_fit = np.asarray(res.conditional_volatility)        # 적합이 되찾은 σ̂_t / the σ̂_t recovered by the fit
print(f"참 σ_t 와 적합 σ̂_t 의 상관 / correlation between the true σ_t and the fitted σ̂_t: {np.corrcoef(sigma_true, sigma_fit)[0, 1]:.3f}")
print(f"둘의 평균 차이 / mean absolute difference: {np.abs(sigma_true - sigma_fit).mean():.3f} %")
print(f"마지막 날 / last day: true {sigma_true[-1]:.3f} %, fitted {sigma_fit[-1]:.3f} %")
# 기대값 / Expected
#   상관 / correlation: 0.995
#   평균 차이 / mean absolute difference: 0.018 %
#   마지막 날 / last day: true 0.845 %, fitted 0.832 %

show = slice(N - 500, N)                                   # 마지막 500일만 / the last 500 days only
plt.figure(figsize=(11, 3.4))
plt.plot(days[show], sigma_true[show], color=GOLD, lw=1.2, label=r"true $\sigma_t$")
plt.plot(days[show], sigma_fit[show], color=GREEN, lw=1.0, ls="--", label=r"fitted $\hat\sigma_t$")
plt.axhline(1.0, color=GRAY, ls=":", lw=0.8); plt.ylabel("volatility (%)"); plt.xlabel("day"); plt.legend()
plt.title("The fit recovers the hidden volatility from the returns alone (last 500 days)", fontsize=10)
plt.tight_layout(); plt.show()

**What to notice.** 수익률만 보고도 숨은 $\sigma_t$ 를 거의 되찾았습니다(상관 0.995). 실제 자료에서는 참 $\sigma_t$ 가 없으니 이런 그림을 그릴 수 없고, 대신 7주차처럼 **잔차**를 점검합니다.
*From the returns alone, the hidden $\sigma_t$ is almost fully recovered (correlation 0.995). On real data there is no true $\sigma_t$ to draw this picture with, so instead we check the **residuals**, as in Week 7.*

### 표준화 잔차 점검 / The standardized residual check

덱 「Did GARCH Catch the Clustering?」: 평균을 빼고 적합한 $\hat\sigma_t$ 로 나눈 표준화 잔차 $z_t$ 가 모형이 맞을 때 독립 N(0, 1) 에 가깝습니다. 그러면 $z_t^2$ 에는 기억이 남지 않아야 합니다. 군집을 $\hat\sigma_t$ 가 다 가져갔는지를 이렇게 봅니다.
*"Did GARCH Catch the Clustering?": subtract the mean and divide by the fitted $\hat\sigma_t$. If the model is right, the standardized residual $z_t$ is close to independent N(0, 1), so $z_t^2$ should keep no memory. That is how we check whether $\hat\sigma_t$ took all the clustering.*

$$z_t = \frac{r_t - \hat\mu}{\hat\sigma_t}$$

**TODO 2 는 이 한 줄입니다.** / *TODO 2 is this one line.*

In [ ]:
# ---------------------------------------------------------------- TODO 2 ----
# 다음 이름을 씁니다 / Names used below
#   r      : 수익률, 길이 n 의 배열 / the returns, an array of length n
#   mu     : 적합한 평균 μ̂ (숫자 하나) / the fitted mean μ̂ (one number)
#   sigma  : 적합한 변동성 σ̂_t, 길이 n 의 배열. 날마다 다르다 / the fitted volatility σ̂_t, an array of length n, different every day
#   z      : 표준화 잔차   <- 여기가 빈칸
#            the standardized residuals   <- this is the blank
#
# 참고 / Reference   배열끼리의 계산은 같은 자리끼리 / arithmetic between arrays works entry by entry
#   예 / e.g.   np.array([2., 4.]) / np.array([1., 2.])  ->  array([2., 2.])

def standardize(r, mu, sigma):
    """수익률을 평균을 빼고 그날의 변동성으로 나눈다 / Subtract the mean and divide each return by that day's volatility."""
    # TODO 2. 덱 「Did GARCH Catch the Clustering?」 의 z_t 식을 그대로 쓰세요. 한 줄입니다.
    #         나누는 것은 분산이 아니라 변동성(표준편차) σ̂_t 입니다.
    # TODO 2. Write the z_t formula on "Did GARCH Catch the Clustering?" as it is. One line.
    #         Divide by the volatility (standard deviation) σ̂_t, not by the variance.
    z = None
    if z is None:
        raise NotImplementedError(BLANK_MSG)

    return z


# 한 번 불러 보기. 손으로 셀 수 있는 두 날 / Call it once, on two days you can check by hand
print(standardize(np.array([1.0, -2.0]), 0.1, np.array([0.9, 1.8])))
# 기대값 / Expected
#   [ 1.     -1.1667]      (1 − 0.1) / 0.9 = 1,  (−2 − 0.1) / 1.8 = −1.1667
#   다른 값이 나와도 바로 아래 자가검증이 어디가 틀렸는지 알려 줍니다.
#   If you get other values, the self-check below tells you what went wrong.
# --------------------------------------------------------------------------

In [ ]:
# 자가검증 / Self-check
got = np.asarray(standardize(np.array([1.0, -2.0]), 0.1, np.array([0.9, 1.8])))
want = np.array([1.0, -2.1 / 1.8])
assert np.allclose(got, want, atol=1e-6), (
    f"정답은 {want.round(4)}, 결과는 {got.round(4)} 입니다. "
    "[1.1111 -1.1111] 이면 평균 0.1 을 빼지 않았고, [1.2346 -0.6481] 이면 σ 대신 σ² 으로 나눴습니다. "
    "(r − μ) / σ 입니다.\n"
    f"Expected {want.round(4)}, got {got.round(4)}. "
    "[1.1111 -1.1111] means the mean 0.1 was not subtracted; [1.2346 -0.6481] means you divided by σ² instead of σ. "
    "The line is (r − μ) / σ.")
print("통과. 평균을 뺀 뒤 그날의 변동성으로 나누면, 모형이 맞을 때 z 는 날마다 같은 폭 1 을 갖습니다.")
print("Passed. After subtracting the mean and dividing by that day's volatility, z has the same spread 1 every day when the model is right.")

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 실행한 뒤에는 바로 위의 자가검증 셀을 다시 실행하세요. 자가검증은 지금 메모리에 있는 것을 검사합니다.
# After running it, run the self-check cell just above again. The self-check tests whatever is in memory now.

def standardize(r, mu, sigma):
    """수익률을 평균을 빼고 그날의 변동성으로 나눈다 / Subtract the mean and divide each return by that day's volatility."""
    z = (r - mu) / sigma                 # 빈칸이었던 줄. 평균을 빼고, 그날의 σ̂_t 로 나눈다 / the blank: subtract the mean, divide by that day's σ̂_t
    if z is None:
        raise NotImplementedError(BLANK_MSG)

    return z


print(standardize(np.array([1.0, -2.0]), 0.1, np.array([0.9, 1.8])))
# 기대값 / Expected
#   [ 1.     -1.1667]

In [ ]:
z = standardize(r, est["mu"], sigma_fit)                   # 모의 자료의 표준화 잔차 / the standardized residuals of the simulated returns
print(f"z 의 평균 · 표준편차 / mean and standard deviation of z: {z.mean():+.3f}, {z.std():.3f}")
print(f"시차 1 의 ACF / ACF at lag 1:   r² {acf1(r ** 2):+.3f}   z² {acf1(z ** 2):+.3f}")
print(f"Ljung–Box(10) p:   z {fmt_p(lb_p(z))}    z² {fmt_p(lb_p(z ** 2))}")
# 기대값 / Expected
#   z 의 평균 · 표준편차 / mean and sd: +0.004, 1.000
#   시차 1 의 ACF / ACF at lag 1:   r² +0.259   z² -0.000
#   Ljung–Box(10) p:   z = 0.66    z² = 0.59
#   r² 에 있던 기억이 z² 에서는 사라졌습니다. 두 p 가 모두 0.05 위라 「패턴이 남지 않았다」 를 버릴 이유가 없습니다.
#   The memory that r² had is gone from z²; both p-values are above 0.05, so there is no reason to drop "no pattern left".

acf_z2 = acf(z ** 2, nlags=20, fft=False)[1:]
plt.figure(figsize=(11, 3.4))
plt.bar(np.arange(1, 21) - 0.18, acf_r2, width=0.36, color=GRAY, label="ACF of r²")
plt.bar(np.arange(1, 21) + 0.18, acf_z2, width=0.36, color=GREEN, label="ACF of z²")
plt.axhline(band, color=GOLD, ls="--", lw=0.8); plt.axhline(-band, color=GOLD, ls="--", lw=0.8)
plt.xlabel("lag"); plt.ylabel("correlation"); plt.legend(); plt.title(r"After dividing by $\hat\sigma_t$, the squared values keep almost no memory", fontsize=10)
plt.tight_layout(); plt.show()

**What to notice.** 덱 「Did GARCH Catch the Clustering?」 의 그림과 같습니다. 회색 막대(r²)의 기억이 초록 막대(z²)에서는 사라졌습니다. 모의 자료는 우리가 정규 오차로 만들었으니 z 의 Ljung–Box 도 통과합니다. 덱의 S&P 500 에서는 정규 오차로는 z 가 0.016 으로 걸렸고, 그래서 교재의 AR(1) 평균 + t 오차가 나왔습니다. 실제 자료가 모의 자료와 다른 점이 바로 거기, 두꺼운 꼬리입니다.
*The same picture as "Did GARCH Catch the Clustering?": the memory in the grey bars (r²) is gone from the green bars (z²). We built the simulation with normal errors, so the Ljung–Box test on z passes too. On the deck's S&P 500 the normal-error fit left z at 0.016, which is why the textbook's AR(1) mean with t errors came up. That is exactly where real data differ from the simulation: fat tails.*

---
### 4단계. 내일의 99% VaR / Step 4. Tomorrow's 99% VaR

**코드를 돌리기 전에 예측하세요.** 마지막 날의 $\hat\sigma_t$ 는 0.83% 였습니다. 내일의 1일 99% VaR 은 몇 % 쯤일까요? 장기 퍼짐 1% 일 때보다 클까요, 작을까요?
*Predict before running. The last day's $\hat\sigma_t$ was 0.83%. About how large is tomorrow's 1-day 99% VaR, in %? Larger or smaller than it would be at the long-run spread of 1%?*

덱 「How Much Could We Lose Tomorrow?」 의 식입니다. 내일의 수익률이 평균 $\mu$, 퍼짐 $\sigma_{t+1}$ 인 정규분포라면, 100 일에 하루만 넘는 손실의 문턱은 다음과 같습니다.
*The formula on "How Much Could We Lose Tomorrow?": if tomorrow's return is normal with mean $\mu$ and spread $\sigma_{t+1}$, the loss exceeded on only 1 day in 100 is*

$$\text{VaR}_{99\%} = -(\mu + z_{0.01}\, \sigma_{t+1}), \qquad z_{0.01} = -2.33$$

**TODO 3 은 이 한 줄입니다.** / *TODO 3 is this one line.*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your prediction here. This cell does not need to be run.
my_prediction_4 = ""

In [ ]:
# ---------------------------------------------------------------- TODO 3 ----
# 다음 이름을 씁니다 / Names used below
#   mu     : 내일 수익률의 평균 / the mean of tomorrow's return
#   sigma  : 내일 수익률의 퍼짐 σ_{t+1} / the spread of tomorrow's return, σ_{t+1}
#   Z01    : 표준정규의 1% 점, −2.3263 (준비 셀) / the 1% point of the standard normal, −2.3263 (setup cell)
#   var99  : 1일 99% VaR, 손실을 양수로   <- 여기가 빈칸
#            the 1-day 99% VaR, a loss written as a positive number   <- this is the blank

def var99(mu, sigma):
    """내일의 1일 99% VaR / Tomorrow's 1-day 99% VaR, a loss written as a positive number."""
    # TODO 3. 덱 「How Much Could We Lose Tomorrow?」 의 식을 그대로 쓰세요. 한 줄입니다.
    #         1% 점은 Z01 (음수)이고, 손실은 양수로 돌려줍니다.
    # TODO 3. Write the formula on "How Much Could We Lose Tomorrow?" as it is. One line.
    #         The 1% point is Z01 (negative); return the loss as a positive number.
    var99 = None
    if var99 is None:
        raise NotImplementedError(BLANK_MSG)

    return var99


# 한 번 불러 보기. 덱의 손 예제: σ_{t+1} = 1.33, μ ≈ 0 / Call it once, with the deck's hand example: σ_{t+1} = 1.33, μ ≈ 0
print(f"VaR = {var99(0.0, 1.33):.2f} %")
# 기대값 / Expected
#   VaR = 3.09 %     덱은 z 를 −2.33 으로 반올림해 3.1% 로 적습니다 / the deck rounds z to −2.33 and writes 3.1%
#   다른 값이 나와도 바로 아래 자가검증이 어디가 틀렸는지 알려 줍니다.
#   If you get another value, the self-check below tells you what went wrong.
# --------------------------------------------------------------------------

In [ ]:
# 자가검증 / Self-check
got = var99(0.0, 1.33)
assert abs(got - 3.094) < 0.01, (
    f"σ = 1.33, μ = 0 에서 정답은 3.09, 결과는 {got:.3f} 입니다. "
    "−3.09 면 부호를 뒤집지 않았고(손실은 양수로), 2.19 면 1% 점 대신 5% 점(1.645)을 썼습니다. "
    "VaR = −(μ + Z01 × σ) 입니다.\n"
    f"σ = 1.33, μ = 0 should give 3.09, but gave {got:.3f}. "
    "−3.09 means the sign was not flipped (a loss is positive); 2.19 means you used the 5% point (1.645) instead of the 1% point. "
    "The line is VaR = −(μ + Z01 × σ).")
got2 = var99(0.1, 2.0)
assert abs(got2 - 4.553) < 0.01, (
    f"σ = 2.0, μ = 0.1 에서 정답은 4.55, 결과는 {got2:.3f} 입니다. 평균 μ 는 손실을 그만큼 줄입니다.\n"
    f"σ = 2.0, μ = 0.1 should give 4.55, but gave {got2:.3f}. The mean μ reduces the loss by that much.")
print("통과. VaR 은 내일의 퍼짐 σ_{t+1} 에 2.33 을 곱한 손실 문턱이고, 평균이 양수면 그만큼 줄어듭니다.")
print("Passed. VaR is 2.33 times tomorrow's spread σ_{t+1}, reduced by a positive mean.")

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 실행한 뒤에는 바로 위의 자가검증 셀을 다시 실행하세요. 자가검증은 지금 메모리에 있는 것을 검사합니다.
# After running it, run the self-check cell just above again. The self-check tests whatever is in memory now.

def var99(mu, sigma):
    """내일의 1일 99% VaR / Tomorrow's 1-day 99% VaR, a loss written as a positive number."""
    var99 = -(mu + Z01 * sigma)          # 빈칸이었던 줄. 1% 분위수를 손실로 뒤집는다 / the blank: the 1% quantile, flipped into a loss
    if var99 is None:
        raise NotImplementedError(BLANK_MSG)

    return var99


print(f"VaR = {var99(0.0, 1.33):.2f} %")
# 기대값 / Expected
#   VaR = 3.09 %

In [ ]:
# 내일의 분산: TODO 1 의 한 줄을 적합한 계수로 한 번 더 / Tomorrow's variance: the TODO 1 line once more, with the fitted coefficients
sigma2_next = next_variance(est["omega"], a_hat, b_hat, r[-1] - est["mu"], sigma_fit[-1] ** 2)
sigma_next = np.sqrt(sigma2_next)
fc = res.forecast(horizon=1)                                                   # arch 의 같은 계산 / the same calculation by arch
print(f"내일의 σ_{{t+1}} / tomorrow's σ_{{t+1}}:  by hand {sigma_next:.3f} %,  arch {np.sqrt(fc.variance.iloc[-1, 0]):.3f} %")
print(f"내일의 1일 99% VaR / tomorrow's 1-day 99% VaR: {var99(est['mu'], sigma_next):.2f} %")
print(f"장기 퍼짐 1% 에서의 VaR / VaR at the long-run spread of 1%: {var99(est['mu'], 1.0):.2f} %")
# 기대값 / Expected
#   내일의 σ_{t+1}: by hand 0.851 %, arch 0.851 %     (손계산과 arch 가 같음 / by hand and arch agree)
#   내일의 VaR / tomorrow's VaR: 1.99 %
#   장기 퍼짐에서의 VaR / at the long-run spread: 2.34 %
#   마지막 날들이 조용해서(σ̂ 0.83 %) 문턱도 낮습니다. 폭풍 뒤라면 두 배 넘게 높아집니다. σ_t 가 날마다 바뀌니 VaR 도 날마다 바뀝니다.
#   The last days were calm (σ̂ 0.83 %), so the threshold is low; after a storm it would be more than twice as high. σ_t changes daily, so VaR does too.

**What to notice.** VaR 은 「이 이상은 안 잃는다」 가 아닙니다. 모형 아래에서 100 일에 하루는 이보다 더 잃습니다(「Common Mistakes With Volatility Models」 다섯째 줄). 모의 자료는 정규 오차라 이 1% 가 맞습니다. 실제 수익률은 정규분포를 그대로 따르지 않으므로, 정규분포의 2.33 으로 만든 VaR 가 실제로 100일에 하루 넘는지는 따로 확인해야 합니다(「Your Turn. What Is the 99% VaR When σ Is 2%?」 의 3번).
*VaR is not "we never lose more than this": under the model, 1 day in 100 loses more (row 5 of "Common Mistakes With Volatility Models"). The simulation has normal errors, so that 1% is right here. Real returns do not follow the normal exactly, so whether a VaR built on the normal 2.33 is really crossed on 1 day in 100 has to be checked separately (question 3 of "Your Turn. What Is the 99% VaR When σ Is 2%?").*

---
### 5단계. 열흘 뒤의 변동성 / Step 5. Volatility ten days ahead

**코드를 돌리기 전에 예측하세요.** 오늘 $\hat\sigma$ 가 0.83% 로 장기 수준 1% 보다 낮습니다. 열흘 뒤의 예측 변동성은 더 내려갈까요, 올라갈까요? 어느 수준으로 다가갈까요?
*Predict before running. Today's $\hat\sigma$ is 0.83%, below the long-run level of 1%. Will the ten-day-ahead forecast go further down or come up? Which level does it approach?*

덱 「Where Will Volatility Go Next?」 의 규칙: 내일의 분산에서 출발해 날마다 $\alpha_1 + \beta_1$ 배씩 장기 수준 쪽으로 갑니다. 6주차 AR(1) 평균 예측과 같은 꼴입니다.
*The rule on "Where Will Volatility Go Next?": start from tomorrow's variance and move toward the long-run level by a factor $\alpha_1 + \beta_1$ each day, the same shape as the Week 6 AR(1) forecast.*

$$\hat\sigma^2_{t+h} = \bar\sigma^2 + (\alpha_1 + \beta_1)^{h-1}\big(\hat\sigma^2_{t+1} - \bar\sigma^2\big), \qquad \bar\sigma^2 = \frac{\alpha_0}{1 - \alpha_1 - \beta_1}$$

**TODO 4 는 이 한 줄입니다.** / *TODO 4 is this one line.*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your prediction here. This cell does not need to be run.
my_prediction_5 = ""

In [ ]:
# ---------------------------------------------------------------- TODO 4 ----
# 다음 이름을 씁니다 / Names used below
#   h           : 며칠 앞인가, 1 이 내일 / how many days ahead; 1 is tomorrow
#   sigma2_next : 내일의 분산 σ²_{t+1} / tomorrow's variance σ²_{t+1}
#   long_run    : 장기 분산 v = α₀ / (1 − α₁ − β₁) / the long-run variance v
#   persist     : α₁ + β₁ / α₁ + β₁
#   sigma2_h    : h 일 뒤의 예측 분산   <- 여기가 빈칸
#                 the forecast variance h days ahead   <- this is the blank
#
# 참고 / Reference   x ** (h - 1)
#   x 의 (h − 1) 제곱. h = 1 이면 1 / x to the power h − 1; equal to 1 when h = 1.   예 / e.g.   0.5 ** 2  ->  0.25

def forecast_variance(h, sigma2_next, long_run, persist):
    """h 일 뒤의 예측 분산 / The forecast variance h days ahead."""
    # TODO 4. 덱 「Where Will Volatility Go Next?」 의 규칙을 그대로 쓰세요: 장기 수준 + (α₁ + β₁)^(h−1) × (내일의 분산 − 장기 수준). 한 줄입니다.
    # TODO 4. Write the rule on "Where Will Volatility Go Next?" as it is: long-run level + (α₁ + β₁)^(h−1) × (tomorrow's variance − long-run level). One line.
    sigma2_h = None
    if sigma2_h is None:
        raise NotImplementedError(BLANK_MSG)

    return sigma2_h


# 한 번 불러 보기. 덱 「One Day of GARCH(1,1) by Hand」 의 큰 날 뒤: 내일의 분산 1.77, 장기 분산 0.4, α₁ + β₁ = 0.95
# Call it once, after the big day of "One Day of GARCH(1,1) by Hand": tomorrow's variance 1.77, long-run variance 0.4, α₁ + β₁ = 0.95
for h in (1, 2, 10):
    print(f"h = {h:2d}:  σ² = {forecast_variance(h, 1.77, 0.4, 0.95):.4f}")
# 기대값 / Expected
#   h =  1:  σ² = 1.7700     내일은 내일의 분산 그대로 / tomorrow is tomorrow's variance itself
#   h =  2:  σ² = 1.7015     0.4 + 0.95 × 1.37
#   h = 10:  σ² = 1.2634     0.4 + 0.95⁹ × 1.37. 장기 수준 0.4 쪽으로 천천히 / slowly toward the long-run 0.4
#   다른 값이 나와도 바로 아래 자가검증이 어디가 틀렸는지 알려 줍니다.
#   If you get other values, the self-check below tells you what went wrong.
# --------------------------------------------------------------------------

In [ ]:
# 자가검증 / Self-check
cases = [(1, 1.77), (2, 1.7015), (10, 0.4 + 0.95 ** 9 * 1.37)]
for h, want in cases:
    got = forecast_variance(h, 1.77, 0.4, 0.95)
    assert abs(got - want) < 1e-6, (
        f"h = {h} 에서 정답은 {want:.4f}, 결과는 {got:.4f} 입니다. "
        "h = 1 이 1.77 이 아니면 지수가 h − 1 이 아니라 h 입니다. 값이 0.4 로 바로 떨어지면 (α₁ + β₁)^(h−1) 을 곱하지 않았습니다. "
        "장기 수준 + (α₁ + β₁)^(h−1) × (내일의 분산 − 장기 수준) 입니다.\n"
        f"h = {h} should give {want:.4f}, but gave {got:.4f}. "
        "If h = 1 is not 1.77, the exponent is h instead of h − 1. If the value drops straight to 0.4, the factor (α₁ + β₁)^(h−1) is missing. "
        "The line is long-run level + (α₁ + β₁)^(h−1) × (tomorrow's variance − long-run level).")
print("통과. 내일의 넘침(1.77 − 0.4)이 날마다 0.95 배씩 줄며 장기 수준으로 돌아갑니다.")
print("Passed. Tomorrow's excess (1.77 − 0.4) shrinks by a factor 0.95 each day as the variance returns to the long-run level.")

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 실행한 뒤에는 바로 위의 자가검증 셀을 다시 실행하세요. 자가검증은 지금 메모리에 있는 것을 검사합니다.
# After running it, run the self-check cell just above again. The self-check tests whatever is in memory now.

def forecast_variance(h, sigma2_next, long_run, persist):
    """h 일 뒤의 예측 분산 / The forecast variance h days ahead."""
    sigma2_h = long_run + persist ** (h - 1) * (sigma2_next - long_run)   # 빈칸이었던 줄. 넘친 만큼이 날마다 (α₁ + β₁) 배로 줄어든다
                                                                          # the blank: the excess shrinks by (α₁ + β₁) each day
    if sigma2_h is None:
        raise NotImplementedError(BLANK_MSG)

    return sigma2_h


for h in (1, 2, 10):
    print(f"h = {h:2d}:  σ² = {forecast_variance(h, 1.77, 0.4, 0.95):.4f}")
# 기대값 / Expected
#   h =  1:  σ² = 1.7700
#   h =  2:  σ² = 1.7015
#   h = 10:  σ² = 1.2634

In [ ]:
long_run_hat = est["omega"] / (1 - persist)                                   # 적합한 장기 분산 / the fitted long-run variance
hs = np.arange(1, 11)
by_hand = np.array([forecast_variance(h, sigma2_next, long_run_hat, persist) for h in hs])   # TODO 4 로 열흘 / ten days with TODO 4
by_arch = res.forecast(horizon=10).variance.iloc[-1].to_numpy()               # arch 의 같은 열흘 / the same ten days by arch
print("h    σ by hand   σ by arch   (%)")
for h, a, b in zip(hs, np.sqrt(by_hand), np.sqrt(by_arch)):
    print(f"{h:2d}    {a:.3f}       {b:.3f}")
print(f"둘의 최대 차이 / largest difference: {np.abs(by_hand - by_arch).max():.1e}    장기 퍼짐(적합) / long-run spread (fitted): {np.sqrt(long_run_hat):.3f} %")
# 기대값 / Expected
#   h = 1: 0.851 % → h = 10: 0.895 %, 손계산과 arch 가 같음(차이 1e-10 아래) / by hand and arch agree (difference below 1e-10)
#   장기 퍼짐(적합) / long-run spread (fitted): 0.977 %
#   오늘이 장기 수준보다 조용하니 예측은 올라가며 0.977 쪽으로 갑니다. 폭풍 뒤라면 내려오며 같은 수준으로 갑니다.
#   Today is calmer than the long-run level, so the forecast rises toward 0.977; after a storm it would come down toward the same level.

plt.figure(figsize=(6, 3.2))
plt.plot(hs, np.sqrt(by_hand), "o-", color=GREEN, label="forecast σ (TODO 4)")
plt.axhline(np.sqrt(long_run_hat), color=GRAY, ls="--", lw=0.8, label="fitted long-run spread")
plt.xlabel("days ahead h"); plt.ylabel("volatility (%)"); plt.legend(); plt.title("From today's calm toward the long-run level", fontsize=10)
plt.tight_layout(); plt.show()

**What to notice.** 열흘 뒤의 예측은 장기 수준으로 가되, $\alpha_1 + \beta_1 = 0.956$ 이라 하루에 4.4% 씩만 다가갑니다. 열흘로는 반도 못 갑니다(반감기 15 일). 덱 「How Can Daily Variance Have One Long-Run Level?」 의 두 분산이 여기 다 있습니다. 날마다 바뀌는 $\hat\sigma_t$ 와, 그것이 돌아가는 한 수준 0.977.
*The ten-day forecast heads for the long-run level, but with $\alpha_1 + \beta_1 = 0.956$ it closes only 4.4% of the gap per day; ten days is less than half the way (half-life 15 days). Both variances of "How Can Daily Variance Have One Long-Run Level?" are here: the $\hat\sigma_t$ that changes every day, and the one level 0.977 it returns to.*

---
### 6단계. 선택 하나를 바꾸기 / Step 6. Change one choice

덱 「In the Lab, Fit GARCH and Price the Risk」 의 마지막 줄입니다. 아래 함수는 0 ~ 2단계를 다른 참값 · 다른 표본 크기로 다시 돌립니다. 표의 네 줄과 그 아래 두 줄(같은 참값을 열 가지 난수로)을 보고, 하나를 골라 **차이 하나를 한 문장으로** 설명하세요. 직접 값을 바꿔 한 줄 더 만들어도 좋습니다.
*The last line of "In the Lab, Fit GARCH and Price the Risk". The function below reruns steps 0 to 2 with another truth or sample size. Look at the four rows and the two lines below them (the same truth under ten random seeds), pick one, and **explain one difference in one sentence**. Add a row of your own if you like.*

In [ ]:
def experiment(alpha=0.10, beta=0.85, n=2000, omega=0.05, seed=7):
    """참값과 표본 크기를 바꿔 0 ~ 2단계를 다시 돌린다 / Rerun steps 0 to 2 with another truth and sample size."""
    g = np.random.default_rng(seed)
    e = g.standard_normal(n + 500)
    s2, x = np.empty(n + 500), np.empty(n + 500)
    s2[0] = omega / (1 - alpha - beta); x[0] = np.sqrt(s2[0]) * e[0]
    for t in range(1, n + 500):
        s2[t] = next_variance(omega, alpha, beta, x[t - 1], s2[t - 1])
        x[t] = np.sqrt(s2[t]) * e[t]
    x = x[500:]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                      # 작은 표본에서 arch 의 수렴 경고를 숨긴다 / hide arch's convergence notes on small samples
        f = arch_model(x, mean="Constant", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
    a, b = f.params["alpha[1]"], f.params["beta[1]"]
    return acf1(x ** 2), a, b, a + b, np.log(0.5) / np.log(a + b)

print("참값 / truth                n      r² ACF(1)   α̂₁     β̂₁     α̂₁+β̂₁   반감기 / half-life")
for label, kw in [("α 0.10, β 0.85 (오늘 / today)", dict()),
                  ("α 0.10, β 0.60             ", dict(beta=0.60)),
                  ("α 0.10, β 0.85, n = 300    ", dict(n=300)),
                  ("α 0.05, β 0.94             ", dict(alpha=0.05, beta=0.94))]:
    n = kw.get("n", 2000)
    c1, a, b, s, hl = experiment(**kw)
    true_hl = np.log(0.5) / np.log(kw.get("alpha", 0.10) + kw.get("beta", 0.85))
    print(f"{label}  {n:5d}     {c1:+.3f}    {a:.3f}   {b:.3f}   {s:.3f}    {hl:5.1f} days (truth {true_hl:4.1f})")

# 표본 크기의 효과는 한 번의 추첨으로는 안 보입니다. 같은 참값을 열 가지 난수로 다시 뽑아 α̂₁ + β̂₁ 의 범위를 봅니다.
# One draw cannot show the effect of the sample size. Redraw the same truth with ten random seeds and look at the range of α̂₁ + β̂₁.
for n in (300, 2000):
    sums = [experiment(n=n, seed=seed)[3] for seed in range(1, 11)]
    print(f"n = {n:4d}, 시드 1 ~ 10 / seeds 1 to 10:  α̂₁ + β̂₁ from {min(sums):.2f} to {max(sums):.2f}   (참값 / truth 0.95)")
# 기대값 / Expected   (둘째 자리까지 / to two decimals)
#   α 0.10, β 0.85 (today)      2000    +0.259    0.079   0.877   0.956    15.4 days (truth 13.5)
#   α 0.10, β 0.60              2000    +0.107    0.077   0.596   0.673     1.8 days (truth  1.9)
#   α 0.10, β 0.85, n = 300      300    +0.022    0.061   0.885   0.946    12.5 days (truth 13.5)
#   α 0.05, β 0.94              2000    +0.270    0.041   0.952   0.993    94.7 days (truth 69.0)
#   n =  300: α̂₁ + β̂₁ from 0.67 to 1.00      n = 2000: from 0.92 to 0.98
#   읽는 법 / How to read it
#   β₁ 0.60: 폭풍이 이틀 만에 반으로 줄고, r² 의 ACF 막대도 0.26 에서 0.11 로 작아집니다.
#     β₁ = 0.60: a storm halves in two days, and the lag-1 bar of r² shrinks from 0.26 to 0.11.
#   n = 300: 시드 7 의 한 번은 참값 근처(0.946)지만, 열 번 뽑으면 0.67 에서 1.00 까지 나옵니다. 300일로는 반감기를 거의 모릅니다.
#     r² 의 ACF +0.02 는 띠 ±0.11 안이라 뭉침조차 보이지 않습니다.
#     n = 300: the seed-7 draw lands near the truth (0.946), but ten draws range from 0.67 to 1.00; 300 days say almost nothing about the half-life.
#     The r² ACF of +0.02 is inside its band of ±0.11, so even the clustering is invisible.
#   α₁ 0.05, β₁ 0.94: 합이 0.99 에서 0.993 으로 0.003 틀렸을 뿐인데 반감기는 69 일에서 95 일이 됩니다. 1 에 가까울수록 작은 오차가 크게 번집니다.
#     α₁ = 0.05, β₁ = 0.94: the sum is off by only 0.003 (0.993 against 0.99), yet the half-life moves from 69 to 95 days. Near 1, a small error spreads.

---
### 선택 A. 실제 자료: 덱의 S&P 500 10년 / Optional A. Real data: the deck's ten years of the S&P 500

`arch` 패키지에 들어 있는 S&P 500 자료를 덱과 같은 10년(2006-04-21 ~ 2016-04-20, 2,517 거래일)으로 잘라 같은 적합을 합니다. 「Fitting GARCH(1,1) to Ten Years of the S&P 500」 의 네 숫자와 「How Often Do Extreme Days Happen?」 의 숫자가 그대로 나옵니다. 내려받기 없음.
*The S&P 500 data shipped with `arch`, cut to the deck's ten years (2006-04-21 to 2016-04-20, 2,517 trading days), with the same fit. The four numbers on "Fitting GARCH(1,1) to Ten Years of the S&P 500" and the numbers on "How Often Do Extreme Days Happen?" come out as they are. No download.*

「How Often Do Extreme Days Happen?」 은 하루의 크기를 두 가지로 잽니다. **같은 σ** 는 모든 날을 10년 전체의 표준편차 하나로 나누고, **그날의 σ_t** 는 $z_t = (r_t - \hat\mu)/\hat\sigma_t$, 곧 그날 GARCH 가 예측한 폭 $\hat\sigma_t$ 로 나눕니다(「Is a −4% Return Always Equally Surprising?」 의 잣대). 두 가지로 4 표준편차 밖의 날을 셉니다(19 와 2).
*"How Often Do Extreme Days Happen?" measures the size of each day in two ways. **Same σ** divides every day by one standard deviation of all ten years. **Own σ_t** uses $z_t = (r_t - \hat\mu)/\hat\sigma_t$, dividing each day by the spread $\hat\sigma_t$ that GARCH predicted for that day (the measure on "Is a −4% Return Always Equally Surprising?"). The cell counts the days beyond 4 standard deviations both ways (19 and 2).*

In [ ]:
#@title ▶ Optional A. 덱의 S&P 500 적합과 꼬리 다시 얻기 / Reproduce the deck's S&P 500 fit and tails { display-mode: "form" }
from arch.data import sp500
close = sp500.load()["Adj Close"]                                              # 종가 / closing prices
sp = (100 * np.log(close).diff().dropna())["2006-04-21":"2016-04-20"]          # 로그 수익률(%), 덱과 같은 10년 / log returns in %, the deck's ten years
print(f"거래일 수 / trading days: {len(sp)}   ({sp.index[0].date()} to {sp.index[-1].date()})")
res_sp = arch_model(sp, mean="Constant", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
print("            추정값 / estimate   표준오차 / std. error   덱 / deck")
deck = {"mu": 0.063, "omega": 0.025, "alpha[1]": 0.111, "beta[1]": 0.871}
for name in ("mu", "omega", "alpha[1]", "beta[1]"):
    print(f"{name:9s}   {res_sp.params[name]:8.3f}        {res_sp.std_err[name]:8.3f}          {deck[name]:5.3f}")
ab = res_sp.params["alpha[1]"] + res_sp.params["beta[1]"]
print(f"α₁ + β₁ = {ab:.3f}   반감기 / half-life {np.log(0.5) / np.log(ab):.0f} days   장기 퍼짐 / long-run spread {np.sqrt(res_sp.params['omega'] / (1 - ab)):.2f} %")

# 「How Often Do Extreme Days Happen?」: 4 표준편차 밖의 날을 두 가지 잣대로 센다
# That slide: count the days beyond 4 standard deviations with two measures
zs = (sp - sp.mean()) / sp.std(ddof=0)                                          # 같은 σ: 10년 전체로 표준화 / same σ: standardized over the ten years
z_t = pd.Series(np.asarray(res_sp.std_resid), index=sp.index)                  # 그날의 σ̂_t: (r_t − μ̂) ÷ σ̂_t / each day's own σ̂_t: (r_t − μ̂) ÷ σ̂_t
sig = pd.Series(np.asarray(res_sp.conditional_volatility), index=sp.index)     # 날마다의 σ̂_t (%) / each day's σ̂_t (%)
print(f"4 표준편차 밖인 날, {len(sp)} 거래일 중 / days beyond 4 standard deviations, out of {len(sp)} trading days")
print(f"  정규라면 / a normal curve expects          {2 * norm.sf(4) * len(sp):.2f}")
print(f"  같은 σ / same σ for every day              {(zs.abs() > 4).sum()}   그중 2008년 가을 / of them in autumn 2008: {(zs.abs() > 4)['2008-09':'2008-12'].sum()}")
print(f"  그날의 σ_t / each day's own GARCH σ_t      {(z_t.abs() > 4).sum()}")
for d in z_t.index[z_t.abs() > 4]:                                              # 그날의 σ_t 로도 4 를 넘는 날 / days beyond 4 even with their own σ_t
    print(f"    {d.date()}   r = {sp[d]:+.2f} %   σ_t = {sig[d]:.2f} %   z_t = {z_t[d]:+.2f}")
# 기대값 / Expected   (덱 「Fitting GARCH(1,1) to Ten Years of the S&P 500」 · 「How Often Do Extreme Days Happen?」 과 같음 / as on those two slides)
#   거래일 수 / trading days: 2517
#   mu 0.063 · omega 0.025 · alpha[1] 0.111 · beta[1] 0.871 (표준오차 0.017 · 0.007 · 0.015 · 0.015)
#   α₁ + β₁ = 0.982   반감기 38 days   장기 퍼짐 1.17 %
#   4 표준편차 밖 / beyond 4:   정규 / normal 0.16 · 같은 σ / same σ 19 (2008년 가을 / autumn 2008: 15) · 그날의 σ_t / own σ_t 2
#   남는 2일 / the 2 days:   2007-02-27 (r −3.53 %, σ_t 0.56 %, z_t −6.48) · 2011-08-04 (r −4.90 %, σ_t 1.21 %, z_t −4.09)
#   19일은 모두 σ_t 가 이미 높던 폭풍 주간이라, 그날의 σ_t 로 재면 하나도 4 를 넘지 않습니다. 남는 2일은 다른 날, 덜 출렁이던 때의 큰 하락입니다.
#   All 19 came in stormy weeks, so against their own σ_t none is beyond 4. The 2 are other days: big drops when σ_t was low.
#   2 는 정규라면 기대하는 0.16 보다 많습니다. GARCH 로 군집을 빼도 꼬리가 두껍습니다.
#   2 is more than the 0.16 a normal curve expects: fat tails remain even after GARCH.

---
### 이번 주에 가져갈 것 / What to take away

덱 「What Did We Learn About Tomorrow’s Market?」 의 세 답(「Today in Three Sentences」 의 세 문장)을 이 노트북의 단계로 다시 적었습니다.
*The three answers on "What Did We Learn About Tomorrow’s Market?" (the three sentences of "Today in Three Sentences"), tied to the steps of this notebook.*

1. 수익률의 방향은 맞히기 어렵지만 크기에는 기억이 있습니다(1단계). r 의 ACF 는 띠 안팎의 작은 막대, r² 의 ACF 는 천천히 줄어드는 큰 막대였습니다. GARCH 를 맞추기 전에 이 두 ACF 를 봅니다.
   *The direction of a return is hard to predict, but its size has memory (step 1): small bars around the band for r, large slowly fading bars for r². Look at both ACFs before fitting GARCH.*
2. GARCH(1,1) 은 어제 소식의 몫과 어제 분산에서 넘어온 몫으로 오늘의 분산을 만듭니다(0단계의 한 줄). 적합은 참값 0.95 를 0.956 으로 찾았고(2단계), 되찾은 $\hat\sigma_t$ 는 참 $\sigma_t$ 와 상관 0.995 였습니다(3단계). 표준화 잔차의 제곱에 기억이 남지 않으면 군집을 다 가져간 것입니다.
   *GARCH(1,1) builds today's variance from a share of yesterday's news and a share carried over from yesterday's variance (the one line of step 0). The fit found the true 0.95 as 0.956 (step 2), and the recovered $\hat\sigma_t$ had correlation 0.995 with the true one (step 3). No memory left in the squared standardized residuals means the clustering was taken.*
3. VaR 은 내일의 퍼짐 $\sigma_{t+1}$ 을 손실 문턱으로 바꾼 수이고(4단계), 변동성 예측은 장기 수준으로 $\alpha_1 + \beta_1$ 배씩 돌아갑니다(5단계). 다만 GARCH 가 모든 극단적인 움직임을 설명하지는 않습니다. 선택 A 의 S&P 500 은 그날의 σ_t 로 재어도 4 표준편차 밖이 2일로, 정규분포가 기대하는 0.16일보다 많습니다. 그래서 정규분포의 2.33 으로 만든 VaR 는 따로 확인해야 합니다.
   *VaR turns tomorrow's spread $\sigma_{t+1}$ into a loss threshold (step 4), and the volatility forecast returns to the long-run level by a factor $\alpha_1 + \beta_1$ a day (step 5). Still, GARCH does not explain every extreme move: for the S&P 500 of Optional A, 2 days are still beyond 4 standard deviations of their own σ_t, where a normal curve expects 0.16. So a VaR built on the normal 2.33 has to be checked separately.*

함정과 짝(「Common Mistakes With Volatility Models」): 둘째 줄 「두 ACF 점검 없이 GARCH 적합」 은 1단계, 셋째 줄 「0.95 와 1.02 를 비슷하다고 읽기」 는 6단계의 반감기 표, 다섯째 줄 「VaR 을 보장으로 여기기」 와 여섯째 줄 「꼬리를 보지 않고 2.33 쓰기」 는 4단계와 선택 A 의 꼬리 일수입니다.
*Pitfalls on that slide: row 2 (fitting GARCH without the two-ACF check) is step 1, row 3 (reading 0.95 and 1.02 as about the same) is the half-life table of step 6, rows 5 and 6 (treating VaR as a guarantee, using 2.33 without looking at the tails) are step 4 and the tail counts of Optional A.*

**Your turn.** 6단계에서 고른 한 줄의 차이를 한 문장으로 적으세요. 예: 「β₁ 을 0.60 으로 낮추니 r² 의 ACF(1) 이 0.26 에서 0.11 로 줄고 반감기가 15 일에서 2 일로 짧아졌다. 넘어오는 몫이 작아 폭풍이 바로 끝난다.」
*Write one sentence about the row you chose in step 6. For example: "Lowering β₁ to 0.60 cut the ACF(1) of r² from 0.26 to 0.11 and the half-life from 15 days to 2: little carries over, so a storm ends at once."*

개인 오류는 코드 셀 번호와 마지막 오류 메시지를 함께 수업 게시판에 남기세요. 마지막 셀은 실행 환경을 적는 기록용입니다.
*For an error, post the code cell number and the last error message on the course board. The last cell records the environment.*

References: Shumway & Stoffer, *Time Series Analysis and Its Applications*, 5th ed., §5.3 GARCH Models · [arch documentation, "ARCH Modeling"](https://arch.readthedocs.io/en/latest/univariate/univariate_volatility_modeling.html) · [statsmodels acf](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.acf.html)

In [ ]:
# 연습 공간 / Scratch space

In [ ]:
# 실행 환경 기록. 오류를 게시판에 남길 때 이 출력도 함께 붙여 주세요.
# A record of the environment. Paste this output too when you post an error on the board.
import sys, platform, datetime, matplotlib
print("date        :", datetime.datetime.now().isoformat(timespec="seconds"))
print("python      :", sys.version.split()[0], "on", platform.platform())
print("numpy       :", np.__version__, "· pandas", pd.__version__)
print("statsmodels :", statsmodels.__version__, "· arch", arch.__version__, "· matplotlib", matplotlib.__version__)
print("random      : seed 7 (step 0 and step 6 simulations)")